## Extracting data from nanoindentation maps to feed Multiscale Materials Modelling Projects in Granta MI database
* Author: David Mercier / Yasmine El Gharoussi / Date: 2024-10
* This script is used to extract data from nanoindentation maps saved in the Nanomecommons database (https://nanomecommons.grantami.com/mi/index.aspx)
* The dataflow is the option 4 in the schema below.

<img src="..\doc\pictures\dataflow1.jpg" height="250">
<img src="..\doc\pictures\dataflow2.jpg" height="250">

## Create a Granta MI Session

Import the GRANTA_MIScriptingToolkit package, python libraries, and create a connection to a Granta MI server.

In [1]:
import GRANTA_MIScriptingToolkit as gdl
from GRANTA_MIScriptingToolkit import granta as mpy

import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import norm
from sklearn.mixture import GaussianMixture

usernameVal = 'STK_user'
passwordVal = 'P455w07d!'

session = mpy.connect('https://nanomecommons.grantami.com/mi_servicelayer', user_name=usernameVal, password=passwordVal)

## Get tabular data attribute information

In [2]:
dbKey = "MI_Blank_v2021"
tableName = "Indentation test data"
recordName = "Ni-SiC"

db = session.get_db(db_key=dbKey)
table = db.get_table(tableName)
Indrecord = table.search_for_records_by_name(recordName)[0]
samplesVal = Indrecord.attributes["Mechanical properties matrix"].value[:]
samplesUnit = Indrecord.attributes["Mechanical properties matrix"]

## Define Mechanical Properties of reference (bulk) materials + Analytical results

In [3]:
# from https://doi.org/10.1080/02670844.2018.1433270

# Results for bulk materials
Ni_ratio = 0.8
SiC_ratio = 0.2
E_Ni = 219.7
dE_Ni = 7.2
E_SiC = 343.2
dE_SiC = 50
H_Ni = 3.9
dH_Ni = 0.3
H_SiC = 11.7
dH_SiC = 3.0
#E_Interphase = Ni_ratio*E_Ni + SiC_ratio*E_SiC
E_Interphase = 247.2
dE_Interphase = 8.0
#H_Interphase = Ni_ratio*H_Ni + SiC_ratio*H_SiC
H_Interphase = 5.2
dH_Interphase = 1.3
E_Artefacts = 0
H_Artefacts = 0
unit = 'GPa'

# Define the data
data = {
    'Material': ['Ni', 'SiC', 'Interphase', 'Artefacts'],
    'E': [E_Ni, E_SiC, E_Interphase, E_Artefacts],
    'H': [H_Ni, H_SiC, H_Interphase, H_Artefacts],
    'dE': [dE_Ni, dE_SiC, dE_Interphase, 0],
    'dH': [dH_Ni, dH_SiC, dH_Interphase, 0]
}

# Create the DataFrame
dfRef = pd.DataFrame(data)
print(dfRef)

     Material      E     H    dE   dH
0          Ni  219.7   3.9   7.2  0.3
1         SiC  343.2  11.7  50.0  3.0
2  Interphase  247.2   5.2   8.0  1.3
3   Artefacts    0.0   0.0   0.0  0.0


# 1st option to load and to plot tabular data

In [4]:
columns=['X position (µm)', 'Y position (µm)', 'Z position (µm)', 'Elastic modulus (GPa)', 'Hardness (GPa)', 'Creep (nm)']

df = pd.DataFrame(samplesVal)
df = df.iloc[:, :-1]
for i in range(0, len(df.columns)):
    df[i] = df[i].str[0]
    df = df.rename(columns={i: columns[i]})
    
df = df.fillna(0)
df=df.round(1)
df.plot.scatter(x='Hardness (GPa)', y='Elastic modulus (GPa)', c='DarkBlue')
# Make the 'X position (µm)' and 'Y position (µm)' columns start from 0
df['X position (µm)'] = df['X position (µm)'] - df['X position (µm)'].min()
df['Y position (µm)'] = df['Y position (µm)'] - df['Y position (µm)'].min()
display(df)

IndexError: list index out of range

# 2nd option to load and to plot tabular data

In [ ]:
cleanData = []
for row in samplesVal:
    cleanRow = []
    for x in row:
        try:
            cleanRow.append(x[0])
        except:
            cleanRow.append(0)
    cleanData.append(cleanRow)
    

dfMat = pd.DataFrame(cleanData)
del dfMat[dfMat. columns[-1]]
dfMat.rename(columns = {0:'X coordinate (µm)', 
                        1:'Y coordinate (µm)', 
                        2:'Z coordinate (nm)', 
                        3:'Elastic modulus (GPa)', 
                        4:'Hardness (GPa)', 
                        5:'Creep (nm)'}, inplace = True)

dfMat=dfMat.round(1)
dfMat.plot.scatter(x='Hardness (GPa)', y='Elastic modulus (GPa)', c='DarkBlue')
# Make the 'X position (µm)' and 'Y position (µm)' columns start from 0
dfMat['X coordinate (µm)'] = dfMat['X coordinate (µm)'] - dfMat['X coordinate (µm)'].min()
dfMat['Y coordinate (µm)'] = dfMat['Y coordinate (µm)'] - dfMat['Y coordinate (µm)'].min()
display(dfMat)

## Plot heat map

In [ ]:
# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# plot the elastic modulus correlation heatmap
dFMatPivot_E = dfMat.pivot(index="Y coordinate (µm)", columns="X coordinate (µm)", values="Elastic modulus (GPa)")
sns.heatmap(dFMatPivot_E, vmin=150, vmax=350, cbar_kws={'label': 'Elastic Modulus (GPa)'}, ax=ax1)

# plot the hardness correlation heatmap
dFMatPivot_H = dfMat.pivot(index="Y coordinate (µm)", columns="X coordinate (µm)", values="Hardness (GPa)")
sns.heatmap(dFMatPivot_H, vmin=3, vmax=12, cbar_kws={'label': 'Hardness (GPa)'}, ax=ax2)

plt.tight_layout()
plt.show()
print(dFMatPivot_E)

## Histogram plots and Gaussian deconvolution

In [ ]:
fig = plt.figure()
binsVal=round(dfMat.size/10)

ax1 = fig.add_subplot(221)
ax1 = dfMat['Elastic modulus (GPa)'].plot.kde()
ax11 = fig.add_subplot(223)
ax11 = dfMat['Elastic modulus (GPa)'].plot.hist(bins=binsVal)
ax2 = fig.add_subplot(222)
ax2 = dfMat['Hardness (GPa)'].plot.kde()
ax22 = fig.add_subplot(224)
ax22 = dfMat['Hardness (GPa)'].plot.hist(bins=binsVal)
ax1.title.set_text('Elastic modulus distribution')
ax1.set_xlabel('Elastic modulus values (GPa)')
ax2.title.set_text('Hardness distribution')
ax2.set_xlabel('Hardness values (GPa)')
plt.tight_layout()
plt.show()

In [ ]:
# Extract data from the KDE plot for Elastic modulus
kde_data_em = ax1.get_lines()[0].get_data()
x_values_em = kde_data_em[0]
y_values_em = kde_data_em[1]

# Extract data from the KDE plot for Hardness
kde_data_hardness = ax2.get_lines()[0].get_data()
x_values_hardness = kde_data_hardness[0]
y_values_hardness = kde_data_hardness[1]

# Now you can use x_values_em, y_values_em, x_values_hardness, and y_values_hardness as needed

data = norm.rvs(5,0.4,size=1000) # you can use a pandas series or a list if you want

# Define subplots
fig = plt.figure(figsize=(8, 6))
ax1 = fig.add_subplot(221)
sns.histplot(df['Elastic modulus (GPa)'], kde=True, stat="density", kde_kws=dict(cut=3), ax=ax1)
ax1.set_title('Elastic modulus distribution')
ax1.set_xlabel('Elastic modulus values (GPa)')
plt.savefig('PDF_E.png', bbox_inches='tight')
plt.savefig('PDF_E.pdf', bbox_inches='tight')
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(8, 6))
ax2 = fig.add_subplot(222)
sns.histplot(df['Hardness (GPa)'], kde=True, stat="density", kde_kws=dict(cut=3), ax=ax2)
ax2.set_title('Hardness distribution')
ax2.set_xlabel('Hardness values (GPa)')
plt.savefig('PDF_H.png', bbox_inches='tight')
plt.savefig('PDF_H.pdf', bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
nPhase = 4
label0 = 'Artefacts'
label1 = 'Ni'
label2 = 'Interphase'
label3 = 'SiC'
properties = ['Elastic modulus (GPa)', 'Hardness (GPa)']

from sklearn.mixture import GaussianMixture
import numpy as np

# Initialize an empty DataFrame
dfGMM = pd.DataFrame()

for i in properties:  # Added colon here
    # Fit a Gaussian Mixture Model with n components
    if i == 'Elastic modulus (GPa)':
        if nPhase == 2:
            gmm = GaussianMixture(n_components=nPhase, init_params='kmeans', random_state=0, means_init = np.array([[100], [300]]))
        elif nPhase == 3:
            gmm = GaussianMixture(n_components=nPhase, init_params='kmeans', random_state=0, means_init = np.array([[100], [200], [300]]))
        elif nPhase == 4:
            gmm = GaussianMixture(n_components=nPhase, init_params='kmeans', random_state=0, means_init = np.array([[50], [200], [250], [300]]))
    elif i == 'Hardness (GPa)':
        if nPhase == 2:
            gmm = GaussianMixture(n_components=nPhase, init_params='kmeans', random_state=0, means_init = np.array([[5], [10]]))
        elif nPhase == 3:
            gmm = GaussianMixture(n_components=nPhase, init_params='kmeans', random_state=0, means_init = np.array([[1], [5], [20]]))
        elif nPhase == 4:
            gmm = GaussianMixture(n_components=nPhase, init_params='kmeans', random_state=0, means_init = np.array([[1], [5], [15], [30]]))

    gmm.fit(dfMat[[i]])
    # Get the means, covariances, and weights of the Gaussian distributions
    means = gmm.means_
    covs = gmm.covariances_
    weights = gmm.weights_
    std_devs = np.sqrt(covs)

    if i == 'Elastic modulus (GPa)':
        # Save the means in the DataFrame
        dfGMM['E'] = gmm.means_.flatten()
        dfGMM['dE'] = std_devs.flatten()
    elif i == 'Hardness (GPa)':
        # Save the means in the DataFrame
        dfGMM['H'] = gmm.means_.flatten()
        dfGMM['dH'] = std_devs.flatten()

    from scipy.stats import norm

    dfFit = pd.DataFrame() # Generate the x values
    x_axis = np.linspace(df[i].min(), df[i].max(), 1000)
    dfFit['x_axis'] = x_axis

    # Generate the y values for each component
    y_axis0 = norm.pdf(x_axis, float(means[0][0]), np.sqrt(float(covs[0][0][0]))) * weights[0]
    y_axis0 = np.clip(y_axis0, None, 1) # Clip values in y_axis0 to a maximum of 1
    dfFit['y_axis0'] = y_axis0
    y_axis1 = norm.pdf(x_axis, float(means[1][0]), np.sqrt(float(covs[1][0][0]))) * weights[1]
    y_axis1 = np.clip(y_axis1, None, 1) # Clip values in y_axis1 to a maximum of 1
    dfFit['y_axis1'] = y_axis1
    if nPhase == 3:
        y_axis2 = norm.pdf(x_axis, float(means[2][0]), np.sqrt(float(covs[2][0][0]))) * weights[2]
        y_axis2 = np.clip(y_axis2, None, 1) # Clip values in y_axis2 to a maximum of 1
        dfFit['y_axis2'] = y_axis2
    if nPhase == 4:
        y_axis2 = norm.pdf(x_axis, float(means[2][0]), np.sqrt(float(covs[2][0][0]))) * weights[2]
        y_axis2 = np.clip(y_axis2, None, 1) # Clip values in y_axis2 to a maximum of 1
        dfFit['y_axis2'] = y_axis2
        y_axis3 = norm.pdf(x_axis, float(means[3][0]), np.sqrt(float(covs[3][0][0]))) * weights[3]
        y_axis3 = np.clip(y_axis3, None, 1) # Clip values in y_axis3 to a maximum of 1
        dfFit['y_axis3'] = y_axis3

    # Plot the original histogram
    fig = plt.figure(figsize=(8, 6))
    ax1 = fig.add_subplot(221)
    sns.histplot(dfMat[i], stat="density", kde_kws=dict(cut=3), ax=ax1)

    # Plot the Gaussian components with labels
    ax1.plot(x_axis, y_axis0, color='red', label=label0)
    ax1.plot(x_axis, y_axis1, color='blue', label=label1)
    if nPhase == 3:
        ax1.plot(x_axis, y_axis2, color='green', label=label2)
    if nPhase == 4:
        ax1.plot(x_axis, y_axis2, color='green', label=label2)
        ax1.plot(x_axis, y_axis3, color='yellow', label=label3)

    if i == 'Elastic modulus (GPa)':
        ax1.set_title('Elastic modulus distribution')
        ax1.set_xlabel('Elastic modulus values (GPa)')

        ax1.legend()
        plt.savefig('PDF_E.png', bbox_inches='tight')
        plt.savefig('PDF_E.pdf', bbox_inches='tight')
        plt.tight_layout()
        plt.show()
    elif i == 'Hardness (GPa)':
        ax1.set_title('Hardness distribution')
        ax1.set_xlabel('Hardness values (GPa)')
        ax1.set_ylim([0, 1])

        ax1.legend()
        plt.savefig('PDF_H.png', bbox_inches='tight')
        plt.savefig('PDF_H.pdf', bbox_inches='tight')
        plt.tight_layout()
        plt.show()

    if i == 'Elastic modulus (GPa)':
        dfFitE = dfFit
    elif i == 'Hardness (GPa)':
        dfFitH = dfFit

print(dfGMM)
print(dfFitE)
print(dfFitH)

## K-means clustering analysis

In [ ]:
# K-MEANS CLUSTERING
# Importing Modules
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from yellowbrick.cluster.elbow import kelbow_visualizer
from sklearn.metrics import silhouette_score

numComp = 4
random_stateVal = 42
sizePnt = 50
sizeMean = 100
alphaVal = 0.5
cmapVal = 'viridis'
colorMean = 'red'
markerSym = '+'

dfCluster=dfMat[["Hardness (GPa)", "Elastic modulus (GPa)"]]

#Elbow Method for Finding the Optimal Number of Clusters in K-Means
# Extract the columns from the DataFrame
X = dfCluster.values  # Convert the DataFrame to a numpy array
# Use the quick method and immediately show the figure
kelbow_visualizer(KMeans(random_state=5), X, k=(2, 10))

kmeans = KMeans(n_clusters=numComp, random_state=random_stateVal).fit(dfCluster)
y_pred = kmeans.predict(dfCluster)
dfCluster['Material'] = y_pred
centroidsKmeans = kmeans.cluster_centers_
print(centroidsKmeans)

scoreKM = silhouette_score(dfCluster, y_pred)
dfMat['Material'] = dfCluster['Material']

# Create a new figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Create a colormap with 4 discrete colors
from matplotlib.colors import LinearSegmentedColormap
# Create a colormap with 4 lighter colors
cmap = LinearSegmentedColormap.from_list("", ["lightcoral", "lightgreen", "lightblue", "lightgrey"])

# Plot the data points with color-coded clusters on the first subplot
ax1.scatter(dfCluster['Hardness (GPa)'], dfCluster['Elastic modulus (GPa)'], c=y_pred, s=sizePnt, alpha=alphaVal, cmap=cmap)

# Plot the Gaussian component means on the first subplot
ax1.scatter(centroidsKmeans[:, 0], centroidsKmeans[:, 1], c=colorMean, marker=markerSym, s=sizeMean, label='K-Means')

ax1.set_xlabel('Hardness (GPa)')
ax1.set_ylabel('Elastic modulus (GPa)')
ax1.legend(loc='best')

# Plot the elastic modulus correlation heatmap on the second subplot
dFMatPivot_Phase = dfMat.pivot(index="Y coordinate (µm)", columns="X coordinate (µm)", values="Material")
sns.heatmap(dFMatPivot_Phase, vmin=0, vmax=3, cmap=cmap, cbar_kws={'ticks': [0, 1, 2, 3], 'label': 'Material Phase'}, ax=ax2)

plt.tight_layout()
plt.savefig('K-Means.jpg')
plt.show()

## K-Medoids clustering analysis

In [ ]:
# K-MEDOIDS CLUSTERING
# Importing Modules
from sklearn_extra.cluster import KMedoids

numComp = 4
random_stateVal = 42
sizePnt = 50
sizeMean = 100
alphaVal = 0.5
cmapVal = 'viridis'
colorMean = 'red'
markerSym = '+'

dfCluster=dfMat[["Hardness (GPa)", "Elastic modulus (GPa)"]]

kmedoids = KMedoids(n_clusters=numComp, random_state=random_stateVal).fit(dfCluster)
y_pred = kmedoids.predict(dfCluster)
dfCluster['Material'] = y_pred
centroidsKmedoids = kmedoids.cluster_centers_
print(centroidsKmedoids)

scoreKMed = silhouette_score(dfCluster, y_pred)
dfMat['Material'] = dfCluster['Material']

# Create a new figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Create a colormap with 4 discrete colors
from matplotlib.colors import LinearSegmentedColormap
# Create a colormap with 4 lighter colors
cmap = LinearSegmentedColormap.from_list("", ["lightcoral", "lightgreen", "lightblue", "lightgrey"])

# Plot the data points with color-coded clusters on the first subplot
ax1.scatter(dfCluster['Hardness (GPa)'], dfCluster['Elastic modulus (GPa)'], c=y_pred, s=sizePnt, alpha=alphaVal, cmap=cmap)

# Plot the Gaussian component means on the first subplot
ax1.scatter(centroidsKmedoids[:, 0], centroidsKmedoids[:, 1], c=colorMean, marker=markerSym, s=sizeMean, label='K-Medoids')

ax1.set_xlabel('Hardness (GPa)')
ax1.set_ylabel('Elastic modulus (GPa)')
ax1.legend(loc='best')

# Plot the elastic modulus correlation heatmap on the second subplot
dFMatPivot_Phase = dfMat.pivot(index="Y coordinate (µm)", columns="X coordinate (µm)", values="Material")
sns.heatmap(dFMatPivot_Phase, vmin=0, vmax=3, cmap=cmap, cbar_kws={'ticks': [0, 1, 2, 3], 'label': 'Material Phase'}, ax=ax2)

plt.tight_layout()
plt.savefig('K-Medoids.jpg')
plt.show()

## 3D Gaussian clustering analysis

In [ ]:
numGauss = 4
dfCluster=dfMat[["Hardness (GPa)", "Elastic modulus (GPa)"]]

# Create and fit a Gaussian Mixture Model
gmm = GaussianMixture(n_components=numGauss, random_state=random_stateVal)
gmm.fit(dfCluster)

# Predict cluster assignments for each data point
y_pred = gmm.predict(dfCluster)
dfCluster['Material'] = y_pred

# Get the means and covariances of the Gaussian components
meansGMM = gmm.means_
print(meansGMM)
covariances = gmm.covariances_

scoreGMM = silhouette_score(dfCluster, y_pred)
dfMat['Material'] = dfCluster['Material']

# Create a new figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Create a colormap with 4 discrete colors
from matplotlib.colors import LinearSegmentedColormap
# Create a colormap with 4 lighter colors
cmap = LinearSegmentedColormap.from_list("", ["lightcoral", "lightgreen", "lightblue", "lightgrey"])

# Plot the data points with color-coded clusters on the first subplot
ax1.scatter(dfCluster['Hardness (GPa)'], dfCluster['Elastic modulus (GPa)'], c=y_pred, s=sizePnt, alpha=alphaVal, cmap=cmap)

# Plot the Gaussian component means on the first subplot
ax1.scatter(meansGMM[:, 0], meansGMM[:, 1], c=colorMean, marker=markerSym, s=sizeMean, label='Gaussian Means')

ax1.set_xlabel('Hardness (GPa)')
ax1.set_ylabel('Elastic modulus (GPa)')
ax1.legend(loc='best')

# Plot the elastic modulus correlation heatmap on the second subplot
dFMatPivot_Phase = dfMat.pivot(index="Y coordinate (µm)", columns="X coordinate (µm)", values="Material")
sns.heatmap(dFMatPivot_Phase, vmin=0, vmax=3, cmap=cmap, cbar_kws={'ticks': [0, 1, 2, 3], 'label': 'Material Phase'}, ax=ax2)

plt.tight_layout()
plt.savefig('GMM.jpg')
plt.show()

## Comparison of ML algorithms using Silhouete score, with ref data and 2D GMM results

In [ ]:
dfRef = dfRef.round(1)
print(dfRef)

dfGMM['Material'] = ['Artefacts', 'Ni', 'Interphase', 'SiC']
dfGMM = dfGMM.reindex(columns=['Material', 'E', 'H', 'dE', 'dH'])
dfGMM = dfGMM.round(1)

dfKmeans = pd.DataFrame(centroidsKmeans, columns=['E', 'H'])
dfKmeans['E'], dfKmeans['H'] = dfKmeans['H'], dfKmeans['E']
dfKmeans = dfKmeans.round(1)

dfKmeds = pd.DataFrame(centroidsKmedoids, columns=['E', 'H'])
dfKmeds['E'], dfKmeds['H'] = dfKmeds['H'], dfKmeds['E']
dfKmeds = dfKmeds.round(1)

dfGMM3D = pd.DataFrame(gmm.means_, columns=['E', 'H'])
dfGMM3D['E'], dfGMM3D['H'] = dfGMM3D['H'], dfGMM3D['E']
dfGMM3D = dfGMM3D.round(1)

In [ ]:
#Phase identification using 'distance'  method
from scipy.spatial import distance

# Define a function to find the closest material
def find_closest_material(row):
    distances = dfRef.apply(lambda r: distance.euclidean(r[['E','H']], row[['E','H']]), axis=1)
    return dfRef.loc[distances.idxmin(), 'Material']

# Apply the function to each row in dfGMM3D
dfGMM['Material'] = dfGMM.apply(find_closest_material, axis=1)
dfKmeans['Material'] = dfKmeans.apply(find_closest_material, axis=1)
dfKmeds['Material'] = dfKmeds.apply(find_closest_material, axis=1)
dfGMM3D['Material'] = dfGMM3D.apply(find_closest_material, axis=1)

print(dfGMM)
print(dfKmeans)
print(dfKmeds)
print(dfGMM3D)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create figures
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))

# Combine all dataframes into one with an additional 'Method' column
df_all = pd.concat([
    dfRef.assign(Method='Bulk materials'),
    dfGMM.assign(Method='2D Gaussian Mixture'),
    dfKmeans.assign(Method='K-Means'),
    dfKmeds.assign(Method='K-Medoids'),
    dfGMM3D.assign(Method='3D Gaussian Mixture')
])

# Plot boxplots for 'E'
sns.boxplot(x='Material', y='E', hue='Method', data=df_all, ax=ax1)

# Set title, legends...
ax1.set_title('E values vs Phases distribution')
ax1.set_ylabel('Elastic modulus (GPa)')
ax1.set_xlabel('Material phase')

# Plot boxplots for 'H'
sns.boxplot(x='Material', y='H', hue='Method', data=df_all, ax=ax2)

# Set title, legends...
ax2.set_title('H values vs Phases distribution')
ax2.set_ylabel('Hardness (GPa)')
ax2.set_xlabel('Material phase')

plt.tight_layout()
plt.show()

In [ ]:
print(dfCluster)


## Image clustering with k-means++ for grain merging

In [ ]:
#From https://doi.org/10.1016/j.matdes.2024.112774
#Comparison with SEM image


## Import results into the MI database

In [ ]:
from datetime import datetime
from GRANTA_MIScriptingToolkit import granta as mpy
import pathlib

mi = mpy.connect('https://nanomecommons.grantami.com/mi_servicelayer',  user_name=usernameVal, password=passwordVal)
db = mi.get_db(db_key="MI_Blank_v2021")
test_table = db.get_table('Indentation statistical data')
folderName = 'Python Demo'

now = datetime.now().strftime("%c")
record_name = 'STK Example :{}'.format(now)
record = test_table.path_from(None, tree_path=['', folderName], end_node=record_name)
record
operatorName = 'D. Mercier'
MLmethod = '3D GMM'

MechPropStr = ['Reduced elastic modulus (mean)', 'Hardness (mean)', 'Elastic modulus', 'Y coordinate']

new_file = record.attributes['Other data']
new_pict = record.attributes['Picture']
build_date = record.attributes['Analysis Date']
operator = record.attributes['Operator']
sample_name = record.attributes['Specimen code']
ML_method = record.attributes['ML method used to analyze NI data']
E_val = record.attributes[MechPropStr[0]]
H_val = record.attributes[MechPropStr[1]]

new_file

In [ ]:
#Import files and data into MI

file_object = mpy.File()
filename = 'ClusterAnalysis.pdf'
with open(filename, 'rb') as file_buffer:
    file_object.binary_data = file_buffer.read()
file_object.name = filename
file_object.description = 'This is an example file'
new_file.object = file_object
new_file

picture = pathlib.Path('./GMM.jpg')
new_pict.load(picture)
new_pict

build_date.value = datetime.now()

operator.value = operatorName
sample_name.value = recordName

E_val.value = meansGMM[0,0]
H_val.value = meansGMM[0,1]

ML_method.value = MLmethod

record.set_attributes([build_date, new_file, new_pict, operator, sample_name, E_val, H_val, ML_method])
record = mi.update([record], include_binary_data_in_refresh=True)[0]

#new_pict = record.attributes['Image']
#from IPython.display import Image
#Image(new_pict.value)
#new_pict.save(path='./check.jpg')

In [ ]:
#min-max for each corresponding meta attribute of each mechanical properties
maxminStr = ['Maximum', 'Minimum']

recE = record.attributes[MechPropStr[0]]
funcMax = recE.meta_attributes[maxminStr[0]]
funcMax.value = max(x_values_em)
funcMin = recE.meta_attributes[maxminStr[1]]
funcMin.value = min(x_values_em)
record.set_attributes([recE, funcMax, funcMin])

recH = record.attributes[MechPropStr[1]]
funcMax = recH.meta_attributes[maxminStr[0]]
funcMax.value = max(x_values_hardness)
funcMin = recH.meta_attributes[maxminStr[1]]
funcMin.value = min(x_values_hardness)
record.set_attributes([recH, funcMax, funcMin])

record = mi.update([record])[0]

In [ ]:
#Add probability distribution into the meta attribute of each mechanical properties
recE = record.attributes[MechPropStr[0]]
funcE = recE.meta_attributes['Probability distribution - Reduced elastic modulus']
recH = record.attributes[MechPropStr[1]]
funcH = recH.meta_attributes['Probability distribution - Hardness']
funcH.column_headers

dfFitEall = pd.DataFrame()
dfFitEall['XVal'] = dfFitE['x_axis']
dfFitEall['YVal'] = dfFitE['y_axis0']
dfFitEall['Materials'] = 'Material 1'
for xE, yE, matE in zip(dfFitEall['XVal'], dfFitEall['YVal'], dfFitEall['Materials']):
    funcE.add_point({'Young\'s modulus [GPa]': xE, 'y': yE, 'Material': matE})

dfFitEall = pd.DataFrame()
dfFitEall['XVal'] = dfFitE['x_axis']
dfFitEall['YVal'] = dfFitE['y_axis1']
dfFitEall['Materials'] = 'Material 2'
for xE, yE, matE in zip(dfFitEall['XVal'], dfFitEall['YVal'], dfFitEall['Materials']):
    funcE.add_point({'Young\'s modulus [GPa]': xE, 'y': yE, 'Material': matE})

if nPhase == 3:
    dfFitEall = pd.DataFrame()
    dfFitEall['XVal'] = dfFitE['x_axis']
    dfFitEall['YVal'] = dfFitE['y_axis2']
    dfFitEall['Materials'] = 'Material 3'
    for xE, yE, matE in zip(dfFitEall['XVal'], dfFitEall['YVal'], dfFitEall['Materials']):
        funcE.add_point({'Young\'s modulus [GPa]': xE, 'y': yE, 'Material': matE})

if nPhase == 4:
    dfFitEall = pd.DataFrame()
    dfFitEall['XVal'] = dfFitE['x_axis']
    dfFitEall['YVal'] = dfFitE['y_axis2']
    dfFitEall['Materials'] = 'Material 3'
    for xE, yE, matE in zip(dfFitEall['XVal'], dfFitEall['YVal'], dfFitEall['Materials']):
        funcE.add_point({'Young\'s modulus [GPa]': xE, 'y': yE, 'Material': matE})
    dfFitEall = pd.DataFrame()
    dfFitEall['XVal'] = dfFitE['x_axis']
    dfFitEall['YVal'] = dfFitE['y_axis3']
    dfFitEall['Materials'] = 'Material 4'
    for xE, yE, matE in zip(dfFitEall['XVal'], dfFitEall['YVal'], dfFitEall['Materials']):
        funcE.add_point({'Young\'s modulus [GPa]': xE, 'y': yE, 'Material': matE})

dfFitEall = pd.DataFrame()
dfFitEall['XVal'] = x_values_em
dfFitEall['YVal'] = y_values_em
dfFitEall['Materials'] = 'All data'
for xE, yE, matE in zip(dfFitEall['XVal'], dfFitEall['YVal'], dfFitEall['Materials']):
    funcE.add_point({'Young\'s modulus [GPa]': xE, 'y': yE, 'Material': matE})

for i in range(1, nPhase + 2):
    funcE.series_linestyles[i] = 'Markers'
#funcE.series_linestyles
#funcE.value
record.set_attributes([funcE])
mi.update([record])

dfFitHall = pd.DataFrame()
dfFitHall['XVal'] = dfFitH['x_axis']
dfFitHall['YVal'] = dfFitH['y_axis0']
dfFitHall['Materials'] = 'Material 1'
for xH, yH, matH in zip(dfFitHall['XVal'], dfFitHall['YVal'], dfFitHall['Materials']):
    funcH.add_point({'Hardness [GPa]': xH, 'y': yH, 'Material': matH})

dfFitHall = pd.DataFrame()
dfFitHall['XVal'] = dfFitH['x_axis']
dfFitHall['YVal'] = dfFitH['y_axis1']
dfFitHall['Materials'] = 'Material 2'
for xH, yH, matH in zip(dfFitHall['XVal'], dfFitHall['YVal'], dfFitHall['Materials']):
    funcH.add_point({'Hardness [GPa]': xH, 'y': yH, 'Material': matH})

if nPhase == 3:
    dfFitHall = pd.DataFrame()
    dfFitHall['XVal'] = dfFitH['x_axis']
    dfFitHall['YVal'] = dfFitH['y_axis2']
    dfFitHall['Materials'] = 'Material 3'
    for xH, yH, matH in zip(dfFitHall['XVal'], dfFitHall['YVal'], dfFitHall['Materials']):
        funcH.add_point({'Hardness [GPa]': xH, 'y': yH, 'Material': matH})

if nPhase == 4:
    dfFitHall = pd.DataFrame()
    dfFitHall['XVal'] = dfFitH['x_axis']
    dfFitHall['YVal'] = dfFitH['y_axis2']
    dfFitHall['Materials'] = 'Material 3'
    for xH, yH, matH in zip(dfFitHall['XVal'], dfFitHall['YVal'], dfFitHall['Materials']):
        funcH.add_point({'Hardness [GPa]': xH, 'y': yH, 'Material': matH})

    dfFitHall = pd.DataFrame()
    dfFitHall['XVal'] = dfFitH['x_axis']
    dfFitHall['YVal'] = dfFitH['y_axis3']
    dfFitHall['Materials'] = 'Material 4'
    for xH, yH, matH in zip(dfFitHall['XVal'], dfFitHall['YVal'], dfFitHall['Materials']):
        funcH.add_point({'Hardness [GPa]': xH, 'y': yH, 'Material': matH})

dfFitHall = pd.DataFrame()
dfFitHall['XVal'] = x_values_hardness
dfFitHall['YVal'] = y_values_hardness
dfFitHall['Materials'] = 'All data'
for xH, yH, matH in zip(dfFitHall['XVal'], dfFitHall['YVal'], dfFitHall['Materials']):
    funcH.add_point({'Hardness [GPa]': xH, 'y': yH, 'Material': matH})

for i in range(1, nPhase + 2):
    funcH.series_linestyles[i] = 'Markers'
#funcH.series_linestyles
#funcH.value
record.set_attributes([funcH])
mi.update([record])

In [ ]:
#Add mechanical maps (E vs H)
funcMap = record.attributes[MechPropStr[2]]
funcMap.value

# Create a dictionary that maps the old values to the new values
material_dict = {0: 'Material 1', 1: 'Material 2', 2: 'Material 3', 3: 'Material 4'}  # replace with your actual mapping
# Replace the values in dfMat['Material'] with the new values
dfMat['Material'] = dfMat['Material'].map(material_dict)
# Sort dfMat by the 'Material' column
dfMat = dfMat.sort_values('Material')
# Get the maximum number of materials
max_materials = dfMat['Material'].nunique()

for Emap, Hmap, Matmap in zip(dfMat['Elastic modulus (GPa)'], dfMat['Hardness (GPa)'], dfMat['Material']):
    funcMap.add_point({'y': Emap, 'Hardness [GPa]': Hmap, 'Material': Matmap})
# Set the 'Markers' style for each series up to the maximum number of materials
for i in range(1, max_materials + 1):
    funcMap.series_linestyles[i] = 'Markers'
record.set_attributes([funcMap])
mi.update([record])

In [ ]:
#Add mechanical maps vs XY coordinates
funcXYMap = record.attributes[MechPropStr[3]]
funcXYMap.value

for Ymap, Xmap, Matmap in zip(dfMat['Y coordinate (µm)'], dfMat['X coordinate (µm)'], dfMat['Material']):
    funcXYMap.add_point({'y': Ymap, 'X coordinate [µm]': Xmap, 'Material': Matmap})
# Set the 'Markers' style for each series up to the maximum number of materials
for i in range(1, nPhase + 1):
    funcXYMap.series_linestyles[i] = 'Markers'
record.set_attributes([funcXYMap])
mi.update([record])

## Add link to the test record

In [ ]:
# Add link into "Indentation statistical data"
new_linked_recs = record.links['Indentation test data']
new_linked_recs.add(Indrecord)
record.set_links('Indentation test data', new_linked_recs)

record = mi.update_links([record])[0]
print(record.links)

# Add link into "Indentation test data"
new_linked_recs = Indrecord.links['Indentation statistical data']
new_linked_recs.add(record)
Indrecord.set_links('Indentation statistical data', new_linked_recs)

Indrecord = mi.update_links([Indrecord])[0]
print(Indrecord.links)

## Change meta attributes of folders and records (color...)

In [ ]:
folder = test_table.path_from(None, tree_path=['', folderName])
folder.color = mpy.RecordColor.White
record.color = mpy.RecordColor.White
mi.update([folder, record])